# BARAM 2026 — MLP 시드 앙상블 → v5

v4의 MLP(튜닝 설정, 블렌드 60~70%)는 **단일 시드**라 초기화 분산에 노출. 시드 5개 평균으로:
- 예측 분산 감소 (확실하고 저위험한 소폭 이득의 고전적 방법)
- MLP가 안정되면 최적 블렌드 w가 상승 가능 → w ∈ {0.5,0.6,0.7,0.8} 재스캔

채택 규율: **CV 두 해(2023·2024) 모두 단일시드 최적(w=0.7 기준 2023=0.6011, 2024=0.6139)을 이길 때만.**
채택 시 최종 파이프라인(FICR 후처리) → `submission_v5.csv`. 학습은 MPS.

## 0. 설정 & GBM 폴드 캐시

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS","1")
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd
import torch, torch.nn as nn
torch.set_num_threads(1)
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
import wind_lib as W
from official_metric import group_scores, metric

DEV="mps" if torch.backends.mps.is_available() else "cpu"
SEEDS=[15,0,1,2,3]          # v4 채택 시드 포함 5개
CFG=dict(h=256,depth=3,drop=0.0,lr=0.0015868563457489381,wd=1e-4,bs=256,emb=4)
print("device:",DEV,"| seeds:",SEEDS)

GROUPS=(1,2,3); FR={}; TGT={}; FR_TE={}
for g in GROUPS:
    df,tgt=W.load_train(g); TGT[g]=tgt
    FR[g]=W.add_spatial(W.build(df,g),"train")
    FR_TE[g]=W.add_spatial(W.build(W.load_test(g),g),"test")
BASE_ALL=[c for c in W.feature_cols(FR[1]) if c not in W.SPATIAL_COLS]+["pc_pred_cf"]
V2=W.lean_features(BASE_ALL)+W.SPATIAL_COLS

def lgbm(): return lgb.LGBMRegressor(objective="mae",n_estimators=600,learning_rate=0.03,num_leaves=63,
    min_child_samples=60,subsample=0.8,subsample_freq=1,colsample_bytree=0.7,reg_lambda=1.0,
    random_state=42,n_jobs=1,verbose=-1)
def histgbm(): return HGB(loss="absolute_error",max_iter=600,learning_rate=0.03,max_leaf_nodes=63,
    l2_regularization=1.0,random_state=42)
def gbm_ens(tr,va,cols,tgt):
    lg=lgbm().fit(tr[cols],tr[tgt]); hg=histgbm().fit(tr[cols].to_numpy(),tr[tgt].to_numpy())
    return 0.5*(lg.predict(va[cols])+hg.predict(va[cols].to_numpy()))

YEAR_FOLDS=[([2022],2023),([2022,2023],2024)]
CACHE={}
for tys,vy in YEAR_FOLDS:
    ent={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]; yr=fr.kst_dtm.dt.year
        tr=fr[yr.isin(tys)]; va=fr[yr==vy]
        if len(tr)==0 or len(va)==0: continue
        iso=W.fit_powercurve(tr,tgt,cap)
        ent[g]=dict(tr2=W.with_pc(tr,iso),va2=W.with_pc(va,iso))
        ent[g]["gbm"]=np.clip(gbm_ens(ent[g]["tr2"],ent[g]["va2"],V2,tgt),0,cap)
    CACHE[vy]=ent
    print(f"cached →{vy}")

def fold_total(preds,vy):
    nm=[]; fi=[]
    for g,p in preds.items():
        tgt=TGT[g]; cap=W.CAP[g]; yt=CACHE[vy][g]["va2"][tgt].to_numpy()
        a,b=group_scores(yt,np.clip(p,0,cap),cap); nm.append(a); fi.append(b)
    return 0.5*(1-np.mean(nm))+0.5*np.mean(fi)

device: mps | seeds: [15, 0, 1, 2, 3]


cached →2023


cached →2024


## 1. MLP (concat, v4 설정) & 시드 앙상블 학습기

In [2]:
class MLP(nn.Module):
    def __init__(s,nf,ng=3,h=256,depth=3,drop=0.0,emb=4):
        super().__init__(); s.emb=nn.Embedding(ng,emb)
        layers=[nn.Linear(nf+emb,h),nn.GELU(),nn.Dropout(drop)]
        for _ in range(depth-1): layers+=[nn.Linear(h,h),nn.GELU(),nn.Dropout(drop)]
        layers+=[nn.Linear(h,1)]; s.net=nn.Sequential(*layers)
    def forward(s,x,g): return s.net(torch.cat([x,s.emb(g)],-1)).squeeze(-1)

def train_one(pool_tr,feats,seed):
    torch.manual_seed(seed); np.random.seed(seed)
    pool_tr=pool_tr.sort_values("kst_dtm")
    mu=pool_tr[feats].mean(); sd=pool_tr[feats].std()+1e-8
    X=((pool_tr[feats]-mu)/sd).to_numpy(np.float32)
    y=pool_tr["cf"].to_numpy(np.float32); gid=pool_tr["gid"].to_numpy(np.int64)
    n=len(X); cut=int(n*0.85)
    Xt=torch.tensor(X,device=DEV); yt=torch.tensor(y,device=DEV); gt=torch.tensor(gid,device=DEV)
    net=MLP(len(feats),h=CFG["h"],depth=CFG["depth"],drop=CFG["drop"],emb=CFG["emb"]).to(DEV)
    opt=torch.optim.AdamW(net.parameters(),lr=CFG["lr"],weight_decay=CFG["wd"])
    best=1e9; st=None; pat=0
    for ep in range(120):
        net.train(); perm=np.random.permutation(np.arange(cut))
        for i in range(0,len(perm),CFG["bs"]):
            b=torch.tensor(perm[i:i+CFG["bs"]],device=DEV)
            opt.zero_grad(); ((net(Xt[b],gt[b])-yt[b]).abs().mean()).backward(); opt.step()
        net.eval()
        with torch.no_grad(): vl=(net(Xt[cut:],gt[cut:])-yt[cut:]).abs().mean().item()
        if vl<best-1e-5: best=vl; st={k:v.clone() for k,v in net.state_dict().items()}; pat=0
        else: pat+=1
        if pat>=10: break
    net.load_state_dict(st); return net,(mu,sd)

def predict_one(net,scaler,fr,feats,g,cap):
    mu,sd=scaler
    X=torch.tensor(((fr[feats]-mu)/sd).to_numpy(np.float32),device=DEV)
    gid=torch.full((len(fr),),g-1,dtype=torch.long,device=DEV)
    net.eval()
    with torch.no_grad(): p=net(X,gid).cpu().numpy()
    return np.clip(p,0,1)*cap

def seed_ens_predict(pool_tr, va_frames, feats, seeds=SEEDS):
    """시드별 학습 → 그룹별 예측 평균. va_frames: {g: frame}."""
    acc={g:[] for g in va_frames}
    for sd_ in seeds:
        net,scaler=train_one(pool_tr,feats,sd_)
        for g,fr in va_frames.items():
            acc[g].append(predict_one(net,scaler,fr,feats,g,W.CAP[g]))
    return {g:np.mean(a,axis=0) for g,a in acc.items()}

def make_pool(frames):
    parts=[]
    for g,tr2 in frames.items():
        p=tr2[V2+["kst_dtm"]].copy(); p["cf"]=tr2[TGT[g]]/W.CAP[g]; p["gid"]=g-1; parts.append(p)
    return pd.concat(parts,ignore_index=True)

## 2. CV: 단일시드 vs 시드5 앙상블 (w 스캔)

In [3]:
WEIGHTS=[0.5,0.6,0.7,0.8]
REF={2023:{"single_w0.7":0.6011},2024:{"single_w0.7":0.6139}}   # FiLM 실험의 concat 단일시드 기준

res={}
for vy in (2023,2024):
    pool=make_pool({g:CACHE[vy][g]["tr2"] for g in CACHE[vy]})
    ens_pred=seed_ens_predict(pool,{g:CACHE[vy][g]["va2"] for g in CACHE[vy]},V2)
    res[vy]={w:fold_total({g:(1-w)*CACHE[vy][g]["gbm"]+w*ens_pred[g] for g in CACHE[vy]},vy) for w in WEIGHTS}
    # MLP 앙상블 단독(w=1)도 참고
    res[vy][1.0]=fold_total(ens_pred,vy)
    print(f"→{vy}: "+"  ".join(f"w{w}={res[vy][w]:.4f}" for w in WEIGHTS+[1.0]))

print("\n=== 단일시드(w=0.7) 대비 ===")
best_w=max(WEIGHTS,key=lambda w:0.5*(res[2023][w]+res[2024][w]))
print(f"시드5 최적 w={best_w}: 2023={res[2023][best_w]:.4f} (단일 0.6011), 2024={res[2024][best_w]:.4f} (단일 0.6139)")
adopt=(res[2023][best_w]>0.6011) and (res[2024][best_w]>0.6139)
print("판정:", f"채택 (w={best_w})" if adopt else "기각 → v4 유지")

→2023: w0.5=0.5968  w0.6=0.5987  w0.7=0.5989  w0.8=0.5988  w1.0=0.5975


→2024: w0.5=0.6083  w0.6=0.6087  w0.7=0.6087  w0.8=0.6087  w1.0=0.6092

=== 단일시드(w=0.7) 대비 ===
시드5 최적 w=0.7: 2023=0.5989 (단일 0.6011), 2024=0.6087 (단일 0.6139)
판정: 기각 → v4 유지


## 3. (채택 시) v5 최종: holdout 후처리 → 전체 재학습 → submission_v5.csv

In [4]:
if adopt:
    BLEND=best_w
    def blend_predict(tr_frames):
        pool=make_pool({g:t for g,(t,_) in tr_frames.items()})
        mlp=seed_ens_predict(pool,{g:v for g,(_,v) in tr_frames.items()},V2)
        out={}
        for g,(tr2,va2) in tr_frames.items():
            cap=W.CAP[g]
            gp=np.clip(gbm_ens(tr2,va2,V2,TGT[g]),0,cap)
            out[g]=np.clip((1-BLEND)*gp+BLEND*mlp[g],0,cap)
        return out
    tr_frames={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]
        m_tr=fr.kst_dtm<W.VALID_START; m_va=fr.kst_dtm>=W.VALID_START
        iso=W.fit_powercurve(fr[m_tr],tgt,cap)
        tr_frames[g]=(W.with_pc(fr[m_tr],iso),W.with_pc(fr[m_va],iso))
    val_pred=blend_predict(tr_frames)
    OOF={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; tr2=tr_frames[g][0]
        oof=np.full(len(tr2),np.nan); years=sorted(tr2.kst_dtm.dt.year.unique())
        if len(years)>=2:
            for ty in years:
                m_in=tr2.kst_dtm.dt.year!=ty; m_out=(tr2.kst_dtm.dt.year==ty).to_numpy()
                oof[m_out]=blend_predict({g:(tr2[m_in],tr2[tr2.kst_dtm.dt.year==ty])})[g]
        else:
            n=len(tr2); cut=int(n*0.7)
            oof[cut:]=blend_predict({g:(tr2.iloc[:cut],tr2.iloc[cut:])})[g]
        OOF[g]=oof
        print(f"g{g} OOF done")
    def debias_fit(tr,tgt,oof):
        d=tr.assign(oof=oof); d=d[np.isfinite(d["oof"])].copy(); d["resid"]=d[tgt]-d["oof"]
        d["lb"]=pd.cut(d["lead_h"],bins=[15,21,27,33,40],labels=False,include_lowest=True)
        d["wq"]=pd.qcut(d["gfs_wind_speed_100m_mean"],5,labels=False,duplicates="drop")
        return d.groupby(["lb","wq"])["resid"].mean(), d["resid"].mean()
    def debias_apply(va,pred,tbl,glob):
        v=va.copy()
        v["lb"]=pd.cut(v["lead_h"],bins=[15,21,27,33,40],labels=False,include_lowest=True)
        v["wq"]=pd.qcut(v["gfs_wind_speed_100m_mean"],5,labels=False,duplicates="drop")
        return pred+np.array([tbl.get(k,glob) for k in zip(v["lb"],v["wq"])])
    def nudge_fit(tr,tgt,oof,cap):
        d=tr.assign(oof=oof); d=d[np.isfinite(d["oof"])]
        yt=d[tgt].to_numpy(); yp=d["oof"].to_numpy(); best=(1.0,0.0); bf=-1
        for s in np.linspace(0.90,1.15,26):
            for sh in np.linspace(-0.06,0.06,25)*cap:
                f=group_scores(yt,np.clip(yp*s+sh,0,cap),cap)[1]
                if f>bf: bf=f; best=(s,sh)
        return best
    STORE={}; choice={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; tr2,va2=tr_frames[g]; bp=val_pred[g]
        tbl,glob=debias_fit(tr2,tgt,OOF[g]); s,sh=nudge_fit(tr2,tgt,OOF[g],cap)
        p1=np.clip(debias_apply(va2,bp,tbl,glob),0,cap)
        cand={"P0_none":bp,"P1_debias":p1,"P3_nudge":np.clip(bp*s+sh,0,cap),
              "P5_deb_nudge":np.clip(p1*s+sh,0,cap)}
        STORE[g]=dict(tbl=tbl,glob=glob,nudge=(s,sh))
        rows=[]
        for name,p in cand.items():
            nm,fi=group_scores(va2[tgt].to_numpy(),p,cap); rows.append(dict(post=name,contrib=fi-nm))
        t=pd.DataFrame(rows).set_index("post"); choice[g]=t["contrib"].idxmax()
        print(f"g{g}: {choice[g]}")
    def apply_post(g,frame,pred):
        cap=W.CAP[g]; st=STORE[g]; ch=choice[g]
        if ch=="P0_none": return pred
        if ch=="P1_debias": return np.clip(debias_apply(frame,pred,st["tbl"],st["glob"]),0,cap)
        if ch=="P3_nudge": return np.clip(pred*st["nudge"][0]+st["nudge"][1],0,cap)
        q=np.clip(debias_apply(frame,pred,st["tbl"],st["glob"]),0,cap)
        return np.clip(q*st["nudge"][0]+st["nudge"][1],0,cap)
    ans=pd.DataFrame({f"kpx_group_{g}":tr_frames[g][1][TGT[g]].to_numpy() for g in GROUPS})
    p1df=pd.DataFrame({f"kpx_group_{g}":apply_post(g,tr_frames[g][1],val_pred[g]) for g in GROUPS})
    t1=metric(ans,p1df)
    print(f"\nv5 holdout 총점: {t1[0]:.4f} (1-NMAE {t1[1]:.4f}, FICR {t1[2]:.4f})  cf. v4=0.6463")
    # 제출 생성 (holdout 비교와 무관하게 CV 채택이 근거)
    full_frames={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]
        iso=W.fit_powercurve(FR[g],tgt,cap)
        full_frames[g]=(W.with_pc(FR[g],iso),W.with_pc(FR_TE[g],iso))
    test_pred=blend_predict(full_frames)
    out=W.load_test(1)[["forecast_id","kst_dtm"]].rename(columns={"kst_dtm":"forecast_kst_dtm"})
    for g in GROUPS:
        out[f"kpx_group_{g}"]=apply_post(g,full_frames[g][1],test_pred[g])
    assert out.shape[0]==8760
    for g in GROUPS:
        c=out[f"kpx_group_{g}"]; assert (c>=0).all() and (c<=W.CAP[g]).all() and c.notna().all()
    out.to_csv("submission_v5.csv",index=False); print("saved submission_v5.csv")
    summary=dict(pipeline=f"V2 65 feats · GBM⊕MLP(seed-ens {len(SEEDS)}, w={BLEND}) · FICR postproc",
        seeds=SEEDS, blend_w=float(BLEND), chosen_post=choice,
        cv={str(vy):{str(w):round(res[vy][w],4) for w in WEIGHTS+[1.0]} for vy in (2023,2024)},
        holdout_total=round(float(t1[0]),4), one_minus_nmae=round(float(t1[1]),4), ficr=round(float(t1[2]),4),
        v4_reference=0.6463)
    json.dump(summary,open("v5_summary.json","w"),ensure_ascii=False,indent=2)
    print(json.dumps(summary,ensure_ascii=False,indent=2))
else:
    print("시드 앙상블 기각 → submission_v4.csv 유지")

시드 앙상블 기각 → submission_v4.csv 유지
